# Import Library

In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import optuna


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


/kaggle/input/equity-post-HCT-survival-predictions/sample_submission.csv
/kaggle/input/equity-post-HCT-survival-predictions/data_dictionary.csv
/kaggle/input/equity-post-HCT-survival-predictions/train.csv
/kaggle/input/equity-post-HCT-survival-predictions/test.csv


In [2]:
pd.set_option('display.max_columns', 100000)
pd.set_option('display.max_rows', 40)

## Load Data

In [3]:
train_file_path = "/kaggle/input/equity-post-HCT-survival-predictions/train.csv"
raw_dataset = pd.read_csv(train_file_path).fillna(0)
print(f"Train Dataset Shape is {raw_dataset.shape}")

Train Dataset Shape is (28800, 60)


In [4]:
raw_dataset.head()

,ID,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,vent_hist,renal_issue,pulm_severe,prim_disease_hct,hla_high_res_6,cmv_status,hla_high_res_10,hla_match_dqb1_high,tce_imm_match,hla_nmdp_6,hla_match_c_low,rituximab,hla_match_drb1_low,hla_match_dqb1_low,prod_type,cyto_score_detail,conditioning_intensity,ethnicity,year_hct,obesity,mrd_hct,in_vivo_tcd,tce_match,hla_match_a_high,hepatic_severe,donor_age,prior_tumor,hla_match_b_low,peptic_ulcer,age_at_hct,hla_match_a_low,gvhd_proph,rheum_issue,sex_match,hla_match_b_high,race_group,comorbidity_score,karnofsky_score,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time
0,0,N/A - non-malignant indication,No,0,No,0.0,0.0,No TBI,No,6.0,Bone marrow,No,No,No,IEA,6.0,+/+,0.0,2.0,0,6.0,2.0,No,2.0,2.0,BM,0,0,Not Hispanic or Latino,2016,No,0,Yes,0,2.0,No,0.00,No,2.0,No,9.942,2.0,FKalone,No,M-F,2.0,More than one race,0.0,90.0,No,0,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,42.356
1,1,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,Peripheral blood,No,No,No,AML,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,PB,Intermediate,MAC,Not Hispanic or Latino,2008,No,Positive,No,Permissive,2.0,No,72.29,No,2.0,No,43.705,2.0,Other GVHD Prophylaxis,No,F-F,2.0,Asian,3.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,1.0,4.672
2,2,N/A - non-malignant indication,No,0,No,2.0,8.0,No TBI,No,6.0,Bone marrow,No,No,No,HIS,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,BM,0,0,Not Hispanic or Latino,2019,No,0,Yes,0,2.0,No,0.00,No,2.0,No,33.997,2.0,Cyclophosphamide alone,No,F-M,2.0,More than one race,0.0,90.0,No,Permissive mismatched,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,19.793
3,3,High,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,Bone marrow,No,No,No,ALL,6.0,+/+,10.0,2.0,P/P,6.0,2.0,No,2.0,2.0,BM,Intermediate,MAC,Not Hispanic or Latino,2009,No,Positive,No,Permissive,2.0,No,29.23,No,2.0,No,43.245,2.0,FK+ MMF +- others,No,M-M,2.0,White,0.0,90.0,Yes,Permissive mismatched,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,102.349
4,4,High,No,0,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,No,No,No,MPN,6.0,+/+,10.0,2.0,0,5.0,2.0,No,2.0,2.0,PB,0,MAC,Hispanic or Latino,2018,No,0,Yes,0,2.0,No,56.81,No,2.0,No,29.740,2.0,TDEPLETION +- other,No,M-F,2.0,American Indian or Alaska Native,1.0,90.0,No,Permissive mismatched,Related,MEL,8.0,No,2.0,No,10.0,0.0,16.223


## Preprocess Data

In [5]:
def Preprocess(dataset):
    dataset = dataset.drop('ID', axis=1)
    dataset = pd.get_dummies(dataset)
    dataset = dataset.astype("float64")
    return dataset

In [6]:
dataset = Preprocess(raw_dataset)

In [7]:
target_label = "efs"
x_dataset = dataset.drop(columns=target_label)
x_dataset = dataset.drop(columns="efs_time")
y_dataset = dataset[target_label]



In [8]:
x_dataset.head()

,hla_match_c_high,hla_high_res_8,hla_low_res_6,hla_high_res_6,hla_high_res_10,hla_match_dqb1_high,hla_nmdp_6,hla_match_c_low,hla_match_drb1_low,hla_match_dqb1_low,year_hct,hla_match_a_high,donor_age,hla_match_b_low,age_at_hct,hla_match_a_low,hla_match_b_high,comorbidity_score,karnofsky_score,hla_low_res_8,hla_match_drb1_high,hla_low_res_10,efs,dri_score_0,dri_score_High,dri_score_High - TED AML case <missing cytogenetics,dri_score_Intermediate,dri_score_Intermediate - TED AML case <missing cytogenetics,dri_score_Low,dri_score_Missing disease status,dri_score_N/A - disease not classifiable,dri_score_N/A - non-malignant indication,dri_score_N/A - pediatric,dri_score_TBD cytogenetics,dri_score_Very high,psych_disturb_0,psych_disturb_No,psych_disturb_Not done,psych_disturb_Yes,cyto_score_0,cyto_score_Favorable,cyto_score_Intermediate,cyto_score_Normal,cyto_score_Not tested,cyto_score_Other,cyto_score_Poor,cyto_score_TBD,diabetes_0,diabetes_No,diabetes_Not done,diabetes_Yes,tbi_status_No TBI,tbi_status_TBI + Cy +- Other,"tbi_status_TBI +- Other, -cGy, fractionated","tbi_status_TBI +- Other, -cGy, single","tbi_status_TBI +- Other, -cGy, unknown dose","tbi_status_TBI +- Other, <=cGy","tbi_status_TBI +- Other, >cGy","tbi_status_TBI +- Other, unknown dose",arrhythmia_0,arrhythmia_No,arrhythmia_Not done,arrhythmia_Yes,graft_type_Bone marrow,graft_type_Peripheral blood,vent_hist_0,vent_hist_No,vent_hist_Yes,renal_issue_0,renal_issue_No,renal_issue_Not done,renal_issue_Yes,pulm_severe_0,pulm_severe_No,pulm_severe_Not done,pulm_severe_Yes,prim_disease_hct_AI,prim_disease_hct_ALL,prim_disease_hct_AML,prim_disease_hct_CML,prim_disease_hct_HD,prim_disease_hct_HIS,prim_disease_hct_IEA,prim_disease_hct_IIS,prim_disease_hct_IMD,prim_disease_hct_IPA,prim_disease_hct_MDS,prim_disease_hct_MPN,prim_disease_hct_NHL,prim_disease_hct_Other acute leukemia,prim_disease_hct_Other leukemia,prim_disease_hct_PCD,prim_disease_hct_SAA,prim_disease_hct_Solid tumor,cmv_status_0,cmv_status_+/+,cmv_status_+/-,cmv_status_-/+,cmv_status_-/-,tce_imm_match_0,tce_imm_match_G/B,tce_imm_match_G/G,tce_imm_match_H/B,tce_imm_match_H/H,tce_imm_match_P/B,tce_imm_match_P/G,tce_imm_match_P/H,tce_imm_match_P/P,rituximab_0,rituximab_No,rituximab_Yes,prod_type_BM,prod_type_PB,cyto_score_detail_0,cyto_score_detail_Favorable,cyto_score_detail_Intermediate,cyto_score_detail_Not tested,cyto_score_detail_Poor,cyto_score_detail_TBD,conditioning_intensity_0,conditioning_intensity_MAC,"conditioning_intensity_N/A, F(pre-TED) not submitted",conditioning_intensity_NMA,conditioning_intensity_No drugs reported,conditioning_intensity_RIC,conditioning_intensity_TBD,ethnicity_0,ethnicity_Hispanic or Latino,ethnicity_Non-resident of the U.S.,ethnicity_Not Hispanic or Latino,obesity_0,obesity_No,obesity_Not done,obesity_Yes,mrd_hct_0,mrd_hct_Negative,mrd_hct_Positive,in_vivo_tcd_0,in_vivo_tcd_No,in_vivo_tcd_Yes,tce_match_0,tce_match_Fully matched,tce_match_GvH non-permissive,tce_match_HvG non-permissive,tce_match_Permissive,hepatic_severe_0,hepatic_severe_No,hepatic_severe_Not done,hepatic_severe_Yes,prior_tumor_0,prior_tumor_No,prior_tumor_Not done,prior_tumor_Yes,peptic_ulcer_0,peptic_ulcer_No,peptic_ulcer_Not done,peptic_ulcer_Yes,gvhd_proph_0,gvhd_proph_CDselect +- other,gvhd_proph_CDselect alone,gvhd_proph_CSA + MMF +- others(not FK),"gvhd_proph_CSA + MTX +- others(not MMF,FK)","gvhd_proph_CSA +- others(not FK,MMF,MTX)",gvhd_proph_CSA alone,gvhd_proph_Cyclophosphamide +- others,gvhd_proph_Cyclophosphamide alone,gvhd_proph_FK+ MMF +- others,gvhd_proph_FK+ MTX +- others(not MMF),"gvhd_proph_FK+- others(not MMF,MTX)",gvhd_proph_FKalone,gvhd_proph_No GvHD Prophylaxis,gvhd_proph_Other GVHD Prophylaxis,"gvhd_proph_Parent Q = yes, but no agent",gvhd_proph_TDEPLETION +- other,gvhd_proph_TDEPLETION alone,rheum_issue_0,rheum_issue_No,rheum_issue_Not done,rheum_issue_Yes,sex_match_0,sex_match_F-F,sex_match_F-M,sex_match_M-F,sex_match_M-M,race_group_American Indian or Alaska Native,race_group_Asian,race_g

In [9]:
y_dataset.head()

0    0.0
1    1.0
2    0.0
3    0.0
4    0.0
Name: efs, dtype: float64

In [10]:
x_train, x_valid, y_train, y_valid = train_test_split(x_dataset, y_dataset, test_size=0.2, random_state=50)
print("{} examples in training, {} examples in testing.".format(
    len(x_train), len(x_valid)))

23040 examples in training, 5760 examples in testing.


## Optuna Models

In [11]:
def score_checker(model, x, y):
    return cross_val_score(model, x, y, scoring="roc_auc", cv=4).mean()


In [12]:
def lr_t(trial, x, y):
    l1 = trial.suggest_float("l1", 0.1, 0.9)
    model = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=l1, max_iter=30)
    return score_checker(model, x, y)

In [13]:
def dtree_t(trial, x, y):
    max_depth = trial.suggest_int("max_depth", 1, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    model = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf)
    return score_checker(model, x, y)

In [14]:
def rforest_t(trial, x, y):
    n_estimators = trial.suggest_int("n_estimators", 10, 1000)
    max_depth = trial.suggest_int("max_depth", 1, 100)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 100)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 100)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf, max_features=max_features)
    return score_checker(model, x, y)

In [15]:
def grad_t(trial, x, y):
    n_estimators = trial.suggest_int("n_estimators", 10, 1000)
    max_depth = trial.suggest_int("max_depth", 1, 100)
    r_state = trial.suggest_int("random_state", 1, 100)
    learn_rate = trial.suggest_float("learning_rate", 0.01, 0.99)
    model = GradientBoostingClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=r_state, learning_rate=learn_rate)
    return score_checker(model, x, y)

In [16]:
lr_study = optuna.create_study(direction="maximize")
lr_study.optimize(lambda trial: lr_t(trial, x_train, y_train), n_trials=10) #### eating too much time cant put more or it gonna take hours
print(lr_study.best_params)
print(lr_study.best_value)


[I 2026-01-10 18:22:05,150] A new study created in memory with name: no-name-ed83b782-58a0-4632-92e4-c98c9f6063e7
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[I 2026-01-10 18:22:16,670] Trial 0 finished with value: 0.674618538037062 and parameters: {'l1': 0.21034766749942044}. Best is trial 0 with value: 0.674618538037062.
/usr/local/lib/pyth

{'l1': 0.21034766749942044}
0.674618538037062


### Cant run or it going to eat lots more time, ill just use logistic

In [17]:
'''dtree_study = optuna.create_study(direction="maximize")
dtree_study.optimize(lambda trial: dtree_t(trial, x_train, y_train), n_trials=10000)
print(dtree_study.best_params)
print(dtree_study.best_value)'''

'dtree_study = optuna.create_study(direction="maximize")\ndtree_study.optimize(lambda trial: dtree_t(trial, x_train, y_train), n_trials=10000)\nprint(dtree_study.best_params)\nprint(dtree_study.best_value)'

In [18]:
'''rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(lambda trial: rforest_t(trial, x_train, y_train), n_trials=10000)
print(rf_study.best_params)
print(rf_study.best_value)'''

'rf_study = optuna.create_study(direction="maximize")\nrf_study.optimize(lambda trial: rforest_t(trial, x_train, y_train), n_trials=10000)\nprint(rf_study.best_params)\nprint(rf_study.best_value)'

In [19]:
"""grad_study = optuna.create_study(direction="maximize")
grad_study.optimize(lambda trial: grad_t(trial, x_train, y_train), n_trials=10000)
print(grad_study.best_params)
print(grad_study.best_value)"""

'grad_study = optuna.create_study(direction="maximize")\ngrad_study.optimize(lambda trial: grad_t(trial, x_train, y_train), n_trials=10000)\nprint(grad_study.best_params)\nprint(grad_study.best_value)'

In [20]:
final_lr = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=lr_study.best_params['l1'], max_iter=100)
final_lr.fit(x_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(l1_ratio=0.21034766749942044, penalty='elasticnet',
                   solver='saga')

In [21]:
test_file_path = "/kaggle/input/equity-post-HCT-survival-predictions/test.csv"

test_data = pd.read_csv(test_file_path).fillna(0)
ids = test_data['ID']
test_data = Preprocess(test_data)
test_data = test_data.reindex(columns=dataset.columns, fill_value = 0)
x_test = test_data.drop(columns="efs_time")


preds = final_lr.predict(x_test)
output = pd.DataFrame({'ID': ids,
                       'prediction': preds.squeeze()})

output.head()

,ID,prediction
0,28800,0.0
1,28801,1.0
2,28802,0.0


In [22]:
sample_submission_df = pd.read_csv("/kaggle/input/equity-post-HCT-survival-predictions/sample_submission.csv")
sample_submission_df['prediction'] = final_lr.predict(x_test)
sample_submission_df.to_csv('/kaggle/working/submission.csv', index=False)
sample_submission_df.head()

,ID,prediction
0,28800,0.0
1,28801,1.0
2,28802,0.0
